# notebook_dna_gold

Builds gold DNA tables from silver sources.

## Prerequisites
- `genealogy.silver_person_tag` — DNA role tags per person
- `genealogy.silver_tag` — tag name lookup
- `genealogy.silver_person_source` — person-root citations including DNA match citations
- `genealogy.silver_source` — source title lookup (identifies kit from xref)
- `genealogy.silver_dna_match` — scraped match list (optional; enriches with side/tree info)
- `genealogy.gold_person_branch` — branch per person
- `genealogy.gold_ancestral_proximity` — proximity to researcher

## Tables produced
| Table | Description |
|---|---|
| `gold_person_dna` | One row per person × DNA citation. DNA role + parsed match data. |
| `gold_dna_coverage` | Per-branch DNA coverage summary. Feeds DNA tab and scoring signals. |

## Ancestry DNA source xrefs
These are stable GEDCOM source record IDs identifying each kit.
Update here if source records change.

| kit_id | source_xref |
|---|---|
| `ed_ancestry` | `@S1272999949@` |
| `gillian_ancestry` | `@S1275055303@` |
| `dad_ancestry` | `@S1275093887@` |


In [0]:
# gold_person_dna
# One row per person x DNA citation (a person can have citations from multiple kits).
#
# DNA roles (from silver_person_tag via silver_tag):
#   'DNA Match'          — this person IS a DNA match (typically added to tree as a stub)
#   'Common DNA Ancestor'— a proven common ancestor between researcher and a match
#   'DNA Connection'     — person on the path between a match and the common ancestor
#
# Citation data comes from silver_person_source filtered to the three known
# Ancestry DNA source xrefs. Match name, cM, segments and predicted relationship
# are parsed here from the citation fields.
#
# silver_dna_match is joined on kit_id + match_name to enrich with side,
# tree info and active flag from the scraped match list.
# This join is LEFT — people in the tree without a corresponding scraped row
# (e.g. MyHeritage matches, or matches added before the first scrape) are retained.

from pyspark.sql import functions as F

# ── Kit ID lookup ─────────────────────────────────────────────────────────────
# Maps the three known Ancestry DNA source xrefs to stable kit_id strings.
# Update if new kits are added.
KIT_XREF_MAP = {
    "@S1272999949@": "ed_ancestry",
    "@S1275055303@": "gillian_ancestry",
    "@S1275093887@": "dad_ancestry",
}
kit_mapping_expr = F.create_map(
    *[v for pair in [(F.lit(k), F.lit(v)) for k, v in KIT_XREF_MAP.items()] for v in pair]
)

# ── DNA role tags ─────────────────────────────────────────────────────────────
dna_tags = (
    spark.table("genealogy.silver_person_tag")
    .join(
        spark.table("genealogy.silver_tag").select("tag_gedcom_id", "tag_name"),
        on="tag_gedcom_id",
        how="inner"
    )
    .where(F.col("tag_name").isin("DNA Match", "Common DNA Ancestor", "DNA Connection"))
    .select(
        "person_gedcom_id",
        F.col("tag_name").alias("dna_role")
    )
)

# ── Person-root DNA citations ─────────────────────────────────────────────────
# Filter silver_person_source to Ancestry DNA source xrefs only.
# Parse structured fields from citation_page and citation_text.
#
# citation_page format:  "AncestryDNA Match to <match_name>"
# citation_text format:  "Predicted Relation: <rel>, sharing <cM> cM across <seg> segments."
dna_citations = (
    spark.table("genealogy.silver_person_source")
    .where(F.col("source_xref").isin(list(KIT_XREF_MAP.keys())))
    .withColumn("kit_id", kit_mapping_expr[F.col("source_xref")])
    # Parse match name: strip "AncestryDNA Match to " prefix
    .withColumn(
        "match_name",
        F.when(
            F.col("citation_page").startswith("AncestryDNA Match to "),
            F.regexp_extract(F.col("citation_page"), r"^AncestryDNA Match to (.+)$", 1)
        ).otherwise(F.col("citation_page"))  # keep raw value if format differs
    )
    # Parse cM: "sharing 196.807 cM across..."
    .withColumn(
        "citation_cm",
        F.regexp_extract(F.col("citation_text"), r"sharing ([\d.]+) cM", 1)
         .cast("double")
    )
    # Parse segments: "across 12 segments"
    .withColumn(
        "citation_segments",
        F.regexp_extract(F.col("citation_text"), r"across (\d+) segments", 1)
         .cast("int")
    )
    # Parse predicted relationship: "Predicted Relation: 3rd Cousin, sharing..."
    .withColumn(
        "citation_predicted_relationship",
        F.regexp_extract(F.col("citation_text"), r"Predicted Relation: ([^,]+)", 1)
    )
    # Normalise access date: strip "Accessed: " prefix
    .withColumn(
        "citation_access_date",
        F.regexp_replace(F.col("citation_date"), r"^Accessed:\s*", "")
    )
    .select(
        "person_gedcom_id",
        "kit_id",
        "source_xref",
        "match_name",
        "citation_cm",
        "citation_segments",
        "citation_predicted_relationship",
        "citation_access_date",
        "citation_url",
        "citation_note",
    )
)

# ── Scraped match data ────────────────────────────────────────────────────────
# Join on kit_id + match_name to enrich with side, tree info, active flag.
# LEFT join — tree persons without a scraped match row are retained.
scraped = (
    spark.table("genealogy.silver_dna_match")
    .select(
        "kit_id",
        "match_name",
        F.col("shared_cm").alias("scraped_cm"),
        "side",
        "has_common_ancestor_hint",
        "tree_info",
        "tree_people_count",
        "active",
        "first_seen_date",
        "last_seen_date",
    )
)

citations_enriched = (
    dna_citations
    .join(scraped, on=["kit_id", "match_name"], how="left")
)

# ── Branch and proximity ──────────────────────────────────────────────────────
branch = (
    spark.table("genealogy.gold_person_branch")
    .select("person_gedcom_id", "branch")
)

proximity = (
    spark.table("genealogy.gold_ancestral_proximity")
    .select(
        F.col("person_id").alias("person_gedcom_id"),
        "ancestral_proximity"
    )
)

# ── Assemble gold_person_dna ──────────────────────────────────────────────────
# Start from dna_tags (all tagged people) and left-join citations.
# A person can have a DNA role tag without a citation (e.g. Common DNA Ancestor
# tagged on an ancestor who has no personal citation).
gold_person_dna = (
    dna_tags
    .join(citations_enriched, on="person_gedcom_id", how="left")
    .join(branch,    on="person_gedcom_id", how="left")
    .join(proximity, on="person_gedcom_id", how="left")
    .select(
        "person_gedcom_id",
        "dna_role",
        "kit_id",
        "match_name",
        "citation_cm",
        "citation_segments",
        "citation_predicted_relationship",
        "citation_access_date",
        "citation_url",
        "citation_note",
        "scraped_cm",
        "side",
        "has_common_ancestor_hint",
        "tree_info",
        "tree_people_count",
        "active",
        "first_seen_date",
        "last_seen_date",
        "branch",
        "ancestral_proximity",
    )
)

(
    gold_person_dna
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("genealogy.gold_person_dna")
)

print(f"gold_person_dna written: {spark.table('genealogy.gold_person_dna').count()} rows")
spark.table("genealogy.gold_person_dna").groupBy("dna_role", "kit_id").count().orderBy("dna_role", "kit_id").show()


In [0]:
# gold_dna_coverage
# Per-branch DNA coverage summary. Feeds the DNA tab branch bars,
# the Ancestors tab DNA column, and future scoring signals.
#
# Key metrics:
#   direct_ancestors_total   — all people with ancestral_proximity = 0 in branch
#   direct_ancestors_dna     — those also tagged 'Common DNA Ancestor'
#                              (proven genetic connection to researcher)
#   ancestor_dna_pct         — direct_ancestors_dna / direct_ancestors_total * 100
#   dna_matches_total        — all DNA Match tagged people in branch
#   dna_matches_active       — those with active = TRUE in scraped list
#   dna_matches_unlinked     — active matches in silver_dna_match for this branch
#                              with no corresponding gold_person_dna row
#                              (i.e. not yet added to tree)
#   close_matches_total      — DNA Match people with citation_cm >= 100
#
# Note: dna_matches_unlinked is calculated separately below as it requires
# looking at silver_dna_match rows that have NO tree person.

from pyspark.sql import functions as F

gold_person_dna = spark.table("genealogy.gold_person_dna")
gold_person_branch = spark.table("genealogy.gold_person_branch")
gold_ancestral_proximity = spark.table("genealogy.gold_ancestral_proximity")
silver_dna_match = spark.table("genealogy.silver_dna_match")

# All direct ancestors per branch (ancestral_proximity = 0)
direct_ancestors = (
    gold_ancestral_proximity
    .where(F.col("ancestral_proximity") == 0)
    .join(
        gold_person_branch,
        gold_ancestral_proximity["person_id"] == gold_person_branch["person_gedcom_id"],
        how="left"
    )
    .groupBy("branch")
    .agg(F.countDistinct("person_id").alias("direct_ancestors_total"))
)

# Direct ancestors with Common DNA Ancestor tag per branch
dna_corroborated = (
    gold_person_dna
    .where(
        (F.col("dna_role") == "Common DNA Ancestor") &
        (F.col("ancestral_proximity") == 0)
    )
    .groupBy("branch")
    .agg(F.countDistinct("person_gedcom_id").alias("direct_ancestors_dna"))
)

# DNA Match people per branch (the match stubs added to tree)
match_people = (
    gold_person_dna
    .where(F.col("dna_role") == "DNA Match")
    .groupBy("branch")
    .agg(
        F.countDistinct("person_gedcom_id").alias("dna_matches_total"),
        F.countDistinct(
            F.when(F.col("active") == True, F.col("person_gedcom_id"))
        ).alias("dna_matches_active"),
        F.countDistinct(
            F.when(F.col("citation_cm") >= 100, F.col("person_gedcom_id"))
        ).alias("close_matches_total"),
    )
)

# Unlinked close matches: active scraped matches (>=100cM) with no tree person.
# These are matches that should probably be added to the tree.
# We can't assign branch here (no tree person yet), so we report at total level
# and include in a separate column on the overall row.
linked_match_names = (
    gold_person_dna
    .where(F.col("dna_role") == "DNA Match")
    .select("kit_id", "match_name")
    .distinct()
)

unlinked_close = (
    silver_dna_match
    .where((F.col("active") == True) & (F.col("shared_cm") >= 100))
    .join(linked_match_names, on=["kit_id", "match_name"], how="left_anti")
    .agg(F.count("*").alias("unlinked_close_matches_total"))
)

# Assemble per-branch coverage
# Use all branches from gold_person_branch as the spine
all_branches = gold_person_branch.select("branch").distinct()

gold_dna_coverage = (
    all_branches
    .join(direct_ancestors,  on="branch", how="left")
    .join(dna_corroborated,  on="branch", how="left")
    .join(match_people,      on="branch", how="left")
    .withColumn("direct_ancestors_total",  F.coalesce(F.col("direct_ancestors_total"),  F.lit(0)))
    .withColumn("direct_ancestors_dna",    F.coalesce(F.col("direct_ancestors_dna"),    F.lit(0)))
    .withColumn("dna_matches_total",       F.coalesce(F.col("dna_matches_total"),       F.lit(0)))
    .withColumn("dna_matches_active",      F.coalesce(F.col("dna_matches_active"),      F.lit(0)))
    .withColumn("close_matches_total",     F.coalesce(F.col("close_matches_total"),     F.lit(0)))
    .withColumn(
        "ancestor_dna_pct",
        F.when(
            F.col("direct_ancestors_total") > 0,
            F.round(
                F.col("direct_ancestors_dna") * 100.0 / F.col("direct_ancestors_total"), 1
            )
        ).otherwise(F.lit(0.0))
    )
    .select(
        "branch",
        "direct_ancestors_total",
        "direct_ancestors_dna",
        "ancestor_dna_pct",
        "dna_matches_total",
        "dna_matches_active",
        "close_matches_total",
    )
)

# Add unlinked_close_matches_total as a single scalar cross-joined
# (not branch-specific as unlinked matches have no branch assignment yet)
gold_dna_coverage = gold_dna_coverage.crossJoin(unlinked_close)

(
    gold_dna_coverage
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("genealogy.gold_dna_coverage")
)

print(f"gold_dna_coverage written: {spark.table('genealogy.gold_dna_coverage').count()} rows")
spark.table("genealogy.gold_dna_coverage").show()
